In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers
from tensorflow.keras import models
train_path = "/kaggle/input/datasets/melikechan/cifar100/cifar100/train"
test_path = "/kaggle/input/datasets/melikechan/cifar100/cifar100/test"

IMG_SIZE = 96
BATCH_SIZE = 64
NUM_CLASSES = 100
train_data = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=42
)

test_data = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

class_names = train_data.class_names

print("Number of classes:", len(class_names))
print(class_names)
AUTOTUNE = tf.data.AUTOTUNE

train_data = train_data.prefetch(AUTOTUNE)
test_data = test_data.prefetch(AUTOTUNE)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.1)
])
base_model = tf.keras.applications.EfficientNetV2B0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = data_augmentation(inputs)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.BatchNormalization()(x)

x = layers.Dense(512, activation="relu")(x)

x = layers.Dropout(0.4)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=8,
        restore_best_weights=True,
        mode="max"
    )
]
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=10,
    callbacks=callbacks
)
base_model.trainable = True
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
history_fine = model.fit(
    train_data,
    validation_data=test_data,
    epochs=40,
    callbacks=callbacks
)
loss, accuracy = model.evaluate(test_data)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)
print("Test Accuracy:", accuracy * 100, "%")
model = tf.keras.models.load_model("best_model.keras")
loss, accuracy = model.evaluate(test_data)

print("Best Test Accuracy:", accuracy * 100, "%")
from PIL import Image

image_path = "/kaggle/input/datasets/aryanjha05/assets-cifar-100/apple.jpeg"

img = Image.open(image_path).convert("RGB")

img = img.resize((IMG_SIZE, IMG_SIZE))

img = np.array(img)

img = np.expand_dims(img, axis=0)

prediction = model.predict(img, verbose=0)

predicted_class = np.argmax(prediction[0])

class_name = class_names[predicted_class]

confidence = prediction[0][predicted_class] * 100

print("Predicted Class:", class_name)
print("Confidence:", confidence, "%")